In [1]:
from pathlib import Path
from PIL import Image, ImageOps

# =========================
# HEIC対応（pillow-heif）
# =========================
HEIF_AVAILABLE = False
try:
    import pillow_heif  # pip install pillow-heif
    pillow_heif.register_heif_opener()
    HEIF_AVAILABLE = True
except Exception:
    HEIF_AVAILABLE = False

# =========================
# 設定
# =========================
base_dir = Path(r"C:\Users\yamaz\Documents\GitHub\yamazakilab-ynu.github.io\docs\img")
output_dir = base_dir / "img_resized"
output_dir.mkdir(exist_ok=True)

# 入力拡張子（何でもOK：HEICも含める）
extensions = [".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".webp", ".heic"]

TARGET_WIDTH = 600

print("=== Image Resize + Convert to PNG Start ===")
print(f"📁 base_dir   : {base_dir}")
print(f"📁 output_dir : {output_dir}")
print(f"🧩 HEIC support: {'ON' if HEIF_AVAILABLE else 'OFF (install pillow-heif)'}\n")

# =========================
# メイン処理
# =========================
for img_path in base_dir.rglob("*"):

    # フォルダは除外
    if img_path.is_dir():
        continue

    # ✅ 出力フォルダ配下は絶対に処理しない（無限増殖防止）
    try:
        img_path.relative_to(output_dir)
        continue
    except ValueError:
        pass

    # 画像以外はスキップ
    if img_path.suffix.lower() not in extensions:
        continue

    # HEICなのにサポートが無ければスキップ（インストール案内）
    if img_path.suffix.lower() == ".heic" and not HEIF_AVAILABLE:
        rel = img_path.relative_to(base_dir)
        print(f"❌ HEIC not supported (install pillow-heif): {rel}")
        continue

    # 出力は常にPNG（元の相対パスを維持して .png 化）
    rel = img_path.relative_to(base_dir)
    out_path = (output_dir / rel).with_suffix(".png")
    out_path.parent.mkdir(parents=True, exist_ok=True)

    # ✅ 既存＆最新ならスキップ
    if out_path.exists():
        if out_path.stat().st_mtime >= img_path.stat().st_mtime:
            print(f"⏭ Skipped (up-to-date): {rel}")
            continue

    try:
        with Image.open(img_path) as img:

            # iPhone画像の向き補正（EXIF Orientation）
            img = ImageOps.exif_transpose(img)

            # 600px以下は拡大しない
            if img.width <= TARGET_WIDTH:
                resized = img.copy()
            else:
                w_percent = TARGET_WIDTH / float(img.width)
                new_h = int(img.height * w_percent)
                resized = img.resize((TARGET_WIDTH, new_h), Image.LANCZOS)

            # PNG保存用にモード調整（透過も保持）
            if resized.mode == "P":
                resized = resized.convert("RGBA")
            elif resized.mode not in ("RGB", "RGBA"):
                # 透過が必要か不明でもPNGならRGBAが安全
                resized = resized.convert("RGBA")

            # PNGとして保存（最適化）
            resized.save(out_path, format="PNG", optimize=True)

            print(f"✅ Saved: {out_path}")

    except Exception as e:
        print(f"❌ Error processing {rel}: {e}")

print("\n=== Process Finished ===")


=== Image Resize + Convert to PNG Start ===
📁 base_dir   : C:\Users\yamaz\Documents\GitHub\yamazakilab-ynu.github.io\docs\img
📁 output_dir : C:\Users\yamaz\Documents\GitHub\yamazakilab-ynu.github.io\docs\img\img_resized
🧩 HEIC support: ON

⏭ Skipped (up-to-date): campus1.png
⏭ Skipped (up-to-date): campus2.png
⏭ Skipped (up-to-date): campus3.png
⏭ Skipped (up-to-date): crystals.jpg
⏭ Skipped (up-to-date): crystals1.jpg
⏭ Skipped (up-to-date): crystals2.jpg
⏭ Skipped (up-to-date): crystals3.jpg
⏭ Skipped (up-to-date): crystals4.jpg
⏭ Skipped (up-to-date): crystals5.jpg
⏭ Skipped (up-to-date): exp-caloric.jpg
⏭ Skipped (up-to-date): exp-sp8.jpg
⏭ Skipped (up-to-date): favicon.png
⏭ Skipped (up-to-date): favicon2.png
⏭ Skipped (up-to-date): favicon3.png
⏭ Skipped (up-to-date): favicon4.png
⏭ Skipped (up-to-date): md-0K.png
⏭ Skipped (up-to-date): md-900K.png
⏭ Skipped (up-to-date): md-half.png
⏭ Skipped (up-to-date): md-mix.png
⏭ Skipped (up-to-date): md-mix.webp
⏭ Skipped (up-to-date): m

In [2]:
from pathlib import Path
from PIL import Image, ImageOps

# =========================
# 設定
# =========================
base_dir = Path(r"C:\Users\yamaz\Documents\GitHub\yamazakilab-ynu.github.io\docs\img\test")
output_dir = base_dir / "img_resized"
output_dir.mkdir(exist_ok=True)

TARGET_SIZE = 600

print("=== 600x600 Square Crop Start ===\n")

for img_path in base_dir.rglob("*"):

    # フォルダ除外
    if img_path.is_dir():
        continue

    # 🔥 ここが重要：出力フォルダ配下を完全スキップ
    if output_dir in img_path.parents:
        continue

    if img_path.suffix.lower() not in [".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".webp"]:
        continue

    rel = img_path.relative_to(base_dir)
    out_path = (output_dir / rel).with_suffix(".png")
    out_path.parent.mkdir(parents=True, exist_ok=True)

    try:
        with Image.open(img_path) as img:

            img = ImageOps.exif_transpose(img)

            scale = TARGET_SIZE / min(img.width, img.height)
            new_w = int(img.width * scale)
            new_h = int(img.height * scale)
            img = img.resize((new_w, new_h), Image.LANCZOS)

            left = (new_w - TARGET_SIZE) // 2
            top = (new_h - TARGET_SIZE) // 2
            img = img.crop((left, top, left + TARGET_SIZE, top + TARGET_SIZE))

            if img.mode not in ("RGB", "RGBA"):
                img = img.convert("RGBA")

            img.save(out_path, format="PNG", optimize=True)

            print(f"✅ Saved: {out_path}")

    except Exception as e:
        print(f"❌ Error: {img_path} - {e}")

print("\n=== Finished ===")

=== 600x600 Square Crop Start ===

✅ Saved: C:\Users\yamaz\Documents\GitHub\yamazakilab-ynu.github.io\docs\img\test\img_resized\md-mix.png

=== Finished ===
